# Results Gate Policy

Purpose: tune edge-floor and side-specific policy thresholds, then pick an operating point for live betting.

Use this notebook to answer:
- What edge floor produces stable ROI + CLV?
- Should over/under use different floors?
- Is the gate helping enough to justify stricter filtering?

Edge-floor and recommendation-mix simulator for BET/HOLD policy tuning.

In [1]:
from pathlib import Path
import subprocess
import sys
import polars as pl
from IPython.display import HTML, display

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "production").exists() and (candidate / "src" / "Python").exists():
        ROOT = candidate
        break

sys.path.insert(0, str(ROOT / "src"))
from Python.notebook_analysis_utils import has_over_clv_red_flag

SIM = ROOT / "production" / "ops" / "policy_simulator.py"
OUT_SWEEP = ROOT / "artifacts" / "odds_log" / "policy_scenario_sweep.parquet"


def show_table(df: pl.DataFrame, max_rows: int = 30, height: int = 420):
    pdf = df.to_pandas().round(3)
    if len(pdf) <= max_rows:
        display(pdf)
        return
    table = pdf.to_html(index=False, na_rep="—")
    display(HTML(f"<div style='max-height:{height}px; overflow:auto; border:1px solid #4443; border-radius:6px'>{table}</div>"))


print("repo:", ROOT)
print("simulator:", SIM)

repo: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props
simulator: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\production\ops\policy_simulator.py


In [2]:
thresholds = "0.08,0.10,0.12,0.14,0.16,0.18"
cmd = [sys.executable, str(SIM), "--thresholds", thresholds]
run = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
print(run.stdout)
if run.returncode != 0:
    raise RuntimeError(run.stderr)

--- latest recommendation mix ---
{'recommendation': 'skip', 'oos_reason': None, 'n': 15}
{'recommendation': 'BET', 'oos_reason': None, 'n': 3}
wrote C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\artifacts\odds_log\policy_scenario_sweep.parquet
wrote C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\artifacts\odds_log\policy_scenario_sweep_latest.csv
--- latest scenario rows ---
{'snapshot_utc': '2026-08-20T15:48:59.063120+00:00', 'scope': 'all', 'edge_floor': 0.08, 'n_bets': 176, 'wins': 88, 'losses': 88, 'win_rate': 0.5, 'total_pnl': 98.73201345901191, 'roi': 0.0076986828603799705, 'avg_edge': 0.17388895421370398, 'avg_clv_pp': 0.007540402301390628, 'total_stake': 12824.533137625436}
{'snapshot_utc': '2026-08-20T15:48:59.069354+00:00', 'scope': 'all', 'edge_floor': 0.1, 'n_bets': 159, 'wins': 80, 'losses': 79, 'win_rate': 0.5031446540880503, 'total_pnl': 183.68880592073882, 'roi': 0.015072999782592623, 'avg_edge': 0.18296466581426646, 'avg_clv_pp': 0.0076793936642

In [3]:
if not OUT_SWEEP.exists():
    print("No scenario sweep artifact yet.")
else:
    sweep = pl.read_parquet(OUT_SWEEP)
    latest_ts = sweep.select(pl.col("snapshot_utc").max()).item()
    latest = sweep.filter(pl.col("snapshot_utc") == latest_ts)
    latest_view = latest.sort(["scope", "edge_floor"])
    if "show_table" in globals():
        show_table(latest_view)
    else:
        print(latest_view)

    if "scope" in latest.columns and "n_bets" in latest.columns:
        all_rows = latest.filter(pl.col("scope") == "all")
        if all_rows.height:
            raw_n = int(all_rows.sort("edge_floor").head(1).select("n_bets").item())
            best_n = int(all_rows.sort("edge_floor", descending=True).head(1).select("n_bets").item())
            print(f"raw_n(at lowest floor)={raw_n} best_line_like_n(at highest floor)={best_n}")

    side_view = latest.filter(pl.col("scope").is_in(["over", "under"])) if "scope" in latest.columns else pl.DataFrame()
    if side_view.height:
        side_health = side_view.select([c for c in ["scope", "edge_floor", "roi", "avg_clv_pp", "n_bets"] if c in side_view.columns]).sort(["scope", "edge_floor"])
        print("\nside CLV/ROI health")
        print(side_health)
        side_health_check = side_view.rename({"scope": "side", "avg_clv_pp": "mean_clv_pp"})
        if has_over_clv_red_flag(side_health_check):
            print("RED FLAG: over avg_clv_pp <= 0 for one or more thresholds.")

,snapshot_utc,scope,edge_floor,n_bets,wins,losses,win_rate,total_pnl,roi,avg_edge,avg_clv_pp,total_stake,edge_floor_over,edge_floor_under
0,2026-08-20T15:48:59.169389+00:00,under,0.18,39,22,17,0.564,863.521,0.2,0.25,0.011,4327.687,NaN,NaN



side CLV/ROI health
shape: (1, 5)
┌───────┬────────────┬──────────┬────────────┬────────┐
│ scope ┆ edge_floor ┆ roi      ┆ avg_clv_pp ┆ n_bets │
│ ---   ┆ ---        ┆ ---      ┆ ---        ┆ ---    │
│ str   ┆ f64        ┆ f64      ┆ f64        ┆ i64    │
╞═══════╪════════════╪══════════╪════════════╪════════╡
│ under ┆ 0.18       ┆ 0.199534 ┆ 0.010847   ┆ 39     │
└───────┴────────────┴──────────┴────────────┴────────┘


In [ ]:
# Governance replay risk panel (policy context, not feature bakeoff)
REPLAY_PATH = ROOT / "artifacts" / "odds_log" / "policy_replay_daily.json"

if not REPLAY_PATH.exists():
    print(f"Missing {REPLAY_PATH}. Run production/ops/build_policy_governance_report.py")
else:
    import json

    replay = json.loads(REPLAY_PATH.read_text(encoding="utf-8"))
    scenarios = replay.get("scenarios", []) if isinstance(replay.get("scenarios"), list) else []
    if not scenarios:
        print("No replay scenarios found.")
    else:
        risk_df = pl.DataFrame(scenarios)
        keep = [
            "scenario",
            "n",
            "roi",
            "clv_mean_pp",
            "geo_growth_log_mean",
            "mc_prob_bankroll_floor_breach",
            "mc_prob_drawdown_breach",
            "mc_median_terminal_bankroll",
            "mc_p10_terminal_bankroll",
        ]
        risk_view = risk_df.select([c for c in keep if c in risk_df.columns]).sort("scenario")
        print("Policy replay risk panel")
        show_table(risk_view, max_rows=20)

In [ ]:
# Regime-aware note: recent windows vs full history
# This controls for production changes in features/calibration/edge floors.
LEDGER_PATH = ROOT / "artifacts" / "odds_log" / "ledger.parquet"
if not LEDGER_PATH.exists():
    print(f"Missing {LEDGER_PATH}")
else:
    led = pl.read_parquet(LEDGER_PATH)
    settled = led.filter(
        (pl.col("status") == "settled")
        & (pl.col("stake").cast(pl.Float64).fill_null(0) > 0)
        & pl.col("edge").is_not_null()
    ).with_columns(
        pl.col("game_date").cast(pl.Utf8).str.slice(0, 10).alias("gdate")
    )

    if settled.is_empty():
        print("No settled rows with stake>0.")
    else:
        latest = settled.select(pl.col("gdate").max()).item()
        windows = [
            ("full_history", None),
            ("last_60_settled", 60),
            ("last_30_settled", 30),
        ]
        rows = []
        sorted_settled = settled.sort("gdate")
        for label, n in windows:
            scope = sorted_settled if n is None else sorted_settled.tail(n)
            stake = float(scope["stake"].cast(pl.Float64).sum())
            pnl = float(scope["pnl"].cast(pl.Float64).sum())
            clv = float(scope["clv_pp"].cast(pl.Float64).mean()) if "clv_pp" in scope.columns and scope.height else None
            rows.append({
                "window": label,
                "asof_game_date": latest,
                "n": int(scope.height),
                "stake": stake,
                "pnl": pnl,
                "roi": (pnl / stake) if stake > 0 else None,
                "mean_clv_pp": clv,
            })
        print("Regime-aware aggregate check (interpret full-history with caution):")
        show_table(pl.DataFrame(rows), max_rows=10)

In [ ]:
# Policy edge-decile realization (current settled policy context)
if not LEDGER_PATH.exists():
    print(f"Missing {LEDGER_PATH}")
else:
    led = pl.read_parquet(LEDGER_PATH)
    settled = led.filter(
        (pl.col("status") == "settled")
        & (pl.col("stake").cast(pl.Float64).fill_null(0) > 0)
        & pl.col("edge").is_not_null()
    ).with_columns(
        pl.col("edge").cast(pl.Float64).alias("edge_f")
    )
    if settled.height < 20:
        print("Not enough settled rows for decile analysis.")
    else:
        dec = (
            settled.with_columns(
                pl.col("edge_f")
                .rank(method="ordinal")
                .mul(10)
                .truediv(pl.lit(float(settled.height)))
                .ceil()
                .clip(1, 10)
                .cast(pl.Int64)
                .alias("edge_decile")
            )
            .group_by("edge_decile")
            .agg(
                pl.len().alias("n"),
                pl.col("stake").cast(pl.Float64).sum().alias("stake"),
                pl.col("pnl").cast(pl.Float64).sum().alias("pnl"),
                pl.col("edge_f").mean().alias("mean_edge"),
                pl.col("clv_pp").cast(pl.Float64).mean().alias("mean_clv_pp"),
            )
            .with_columns(
                pl.when(pl.col("stake") > 0).then(pl.col("pnl") / pl.col("stake")).otherwise(None).alias("roi")
            )
            .sort("edge_decile")
        )
        print("Edge-decile realization (policy context)")
        show_table(dec, max_rows=20)